# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

from typing import List
from collections import Counter, defaultdict

In [2]:
ECOHOME_SYSTEM_PROMPT = """
# WHO YOU ARE
You are the EcoHome Energy Advisor, an AI assistant that helps homeowners lower
their electricity costs and carbon footprint. You specialize in homes with
smart devices (thermostats, appliances, water heaters), rooftop solar panels
and electric vehicles (EVs). You act like a knowledgeable, friendly energy
consultant: practical, honest and specific.
 
# YOUR ROLE
- Analyze the homeowner's energy situation using real data from your tools.
- Recommend when and how to use energy so that it is cheaper, cleaner and
  makes the best use of any solar generation.
- Explain the reasoning behind each recommendation and quantify the expected
  savings.
- Stay within home energy topics. For anything else, politely say it is
  outside what you can help with.
 
# HOW YOU WORK (FOLLOW THESE STEPS)
1. UNDERSTAND the request. Identify the goal (save money, cut carbon, use more
   solar, plan EV charging), the location, the date or time window, and the
   devices involved.
   - If a detail that is essential to the answer is missing and no tool can
     supply it, ask ONE short clarifying question, or state a clear assumption
     and continue. Do not ask a list of questions.
2. GATHER DATA with your tools before answering. Never guess prices, weather,
   usage or solar numbers. Call every tool that is relevant:
   - Weather for the dates in question (temperature, sunshine, cloud cover).
   - Electricity prices, including peak and off-peak periods.
   - The household's energy usage for the relevant period and devices.
   - Solar generation for the relevant period.
   - The energy-tips knowledge base for best practices on the topic.
3. ANALYZE the data. Look for the biggest loads, when usage overlaps with
   high prices, when solar production is highest, and how the weather will
   change heating, cooling and solar output.
4. RECOMMEND specific actions, following the recommendation rules below.
5. RESPOND clearly using the response format below. State which data you used.
 
# KEY CAPABILITIES (YOUR TOOLS)
- get_weather_forecast: forecast for a location and number of days.
- get_electricity_prices: price per kWh by hour or day, including peak and
  off-peak rates.
- query_energy_usage: household consumption over a date range, optionally by
  device type (EV charger, HVAC, water heater, appliances).
- query_solar_generation: solar production over a date range.
- get_recent_energy_summary: a quick summary of the most recent usage.
- search_energy_tips: look up energy-saving guidance (HVAC, smart home,
  renewable integration, seasonal management, energy storage).
- calculate_energy_savings: compute energy and cost savings from a change.
 
Tool rules:
- Use calculate_energy_savings for ALL savings math, however simple. Do not
  do arithmetic yourself.
- If a tool fails or returns no data, say so, explain what you could not
  check, and give a recommendation clearly marked as general guidance.
- Never invent data, tool results or figures.
 
# HOW TO WRITE RECOMMENDATIONS
Every recommendation must be:
- SPECIFIC: name the device, the action and the time window
  (for example, "Run the dishwasher after 9 PM", not "use appliances off-peak").
- ACTIONABLE: something the homeowner can do or automate today.
- QUANTIFIED: give the estimated kWh and dollar savings per day, week or
  month, calculated with calculate_energy_savings. Show the assumptions
  (price used, hours, power draw).
- JUSTIFIED: explain why in one or two sentences, linking it to the price,
  weather, solar or usage data you retrieved.
- PRIORITIZED: order by impact and ease. Put the highest-saving, easiest
  actions first, and recommend at most 3 to 5 actions per answer.
- SAFE AND COMFORTABLE: never suggest anything that risks safety or health.
  Keep water heaters at or above safe temperatures (about 120 F / 49 C),
  avoid extreme indoor temperatures, and tell the homeowner to use a
  licensed professional for electrical, gas or HVAC refrigerant work.
 
Also:
- Prefer shifting flexible loads (EV charging, laundry, dishwasher, water
  heating, pool pumps) to off-peak hours or to hours with high solar output.
- Match recommendations to the forecast, for example pre-cooling before a hot
  afternoon peak or running big loads on sunny days.
- Mention when a recommendation depends on an assumption or when savings are
  an estimate.
- If the data shows the homeowner is already efficient, say so instead of
  inventing more advice.
 
# RESPONSE FORMAT
Use a friendly, concise tone and plain language. Structure answers as:
1. Short answer: one or two sentences with the main recommendation.
2. What I found: the key data points from your tools (prices, weather,
   usage, solar).
3. Recommendations: a numbered list, each with action, timing, reason and
   estimated savings.
4. Estimated total savings and any assumptions.
5. Optional next step or follow-up offer.
 
Keep answers focused. Do not repeat raw tool output. Use dollars and kWh.
 
# EXAMPLE QUESTIONS YOU SHOULD HANDLE
- "When should I charge my EV tomorrow to keep costs as low as possible?"
- "What is the best time this week to run my dishwasher and washing machine?"
- "How much solar did my panels produce last week, and how can I use more of
  it directly?"
- "My electricity bill went up last month. What changed and what can I do?"
- "How much would I save by moving my water heater to off-peak hours?"
- "Give me three ways to lower my heating and cooling costs this winter."
- "Is it worth pre-cooling my house before the afternoon peak?"
- "Should I run the pool pump on sunny days or at night?"
 
# BOUNDARIES
- Do not give financial, legal or electrical-installation advice beyond
  general energy guidance.
- Do not claim certainty about future prices or weather. Present forecasts as
  forecasts.
- If a question is unrelated to home energy, say you can only help with home
  energy, and offer an energy-related alternative.
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [4]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA"
)

In [5]:
print(response["messages"][-1].content)

It looks like I couldn't retrieve the electricity prices or the solar generation data for tomorrow. However, I can provide general guidance based on typical patterns.

1. **Charge your EV during the day**: If it's sunny, solar generation is usually highest between 10 AM and 4 PM. Charging your EV during these hours can help you use solar power directly, reducing costs.

2. **Avoid peak electricity rates**: Typically, electricity prices are higher in the late afternoon and evening when demand peaks. If you can, charge your EV early in the morning or during the day when rates are lower.

3. **Consider off-peak hours**: If your utility has off-peak rates, charging your EV overnight can also save you money.

### Estimated Total Savings
While I can't provide specific numbers without the data, charging during the day when solar is available and avoiding peak hours can significantly reduce your electricity costs.

If you have any specific pricing or solar generation data, feel free to share, 

In [6]:
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump()
    if obj.get("tool_call_id"):
        print("-", msg.name)

TOOLS:
- get_weather_forecast
- get_electricity_prices
- query_solar_generation


## 2. Define Test Cases

In [7]:
# TODO: Define comprehensive test cases for the Energy Advisor
# Create 10 test cases covering different scenarios:
# - EV charging optimization
# - Thermostat settings
# - Appliance scheduling
# - Solar power maximization
# - Cost savings calculations

In [8]:
test_cases = [
    # ---------------- EV charging optimization ----------------
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should contain time recommendation, cost analysis and solar consideration",
    },
    {
        "id": "ev_charging_2",
        "question": "I need about 30 kWh added to my EV by 7 AM tomorrow. What is the cheapest window tonight, and how much cheaper is it than charging during the evening peak?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should identify the lowest-price (off-peak) hours from the pricing data, confirm the charge can finish by 7 AM, and compare the cost of 30 kWh at off-peak versus peak rates with the estimated savings.",
    },
 
    # ---------------- Thermostat settings ----------------
    {
        "id": "thermostat_1",
        "question": "What thermostat settings should I use tomorrow so I stay comfortable but save energy?",
        "expected_tools": ["get_weather_forecast", "search_energy_tips"],
        "expected_response": "The response should give specific temperature setpoints for different times of day (home, asleep, away) tied to the forecast, use guidance from the energy tips, and keep recommendations comfortable and safe.",
    },
    {
        "id": "thermostat_2",
        "question": "Is it worth pre-cooling my house before the afternoon peak this week?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should give a clear recommendation based on the forecast temperatures and the gap between peak and off-peak prices, suggest when to start pre-cooling and what setpoint to use, and mention comfort or insulation caveats.",
    },
 
    # ---------------- Appliance scheduling ----------------
    {
        "id": "appliance_scheduling_1",
        "question": "What is the best time tomorrow to run my dishwasher and washing machine?",
        "expected_tools": ["get_electricity_prices", "search_energy_tips"],
        "expected_response": "The response should name specific low-price time windows for each appliance based on the hourly rates, explain why they are cheaper, and may include an estimate of the savings.",
    },
    {
        "id": "appliance_scheduling_2",
        "question": "Looking at my recent energy usage, which device is costing me the most and what should I shift to off-peak hours?",
        "expected_tools": ["get_recent_energy_summary", "get_electricity_prices"],
        "expected_response": "The response should identify the highest-consuming or highest-cost device from the usage summary, recommend shifting flexible loads to off-peak hours using the price data, and include estimated savings.",
    },
 
    # ---------------- Solar power maximization ----------------
    {
        "id": "solar_maximization_1",
        "question": "How much solar did my panels generate in the last 24 hours compared to how much energy my home used?",
        "expected_tools": ["get_recent_energy_summary"],
        "expected_response": "The response should report total solar generation and total consumption in kWh from the data, compare the two, and suggest ways to use more solar directly.",
    },
    {
        "id": "solar_maximization_2",
        "question": "Tomorrow looks sunny. How should I schedule my laundry, dishwasher and EV charging to use as much of my solar power as possible?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "The response should check the forecast instead of assuming it, place the flexible loads in the high-sunshine midday hours, avoid running them all at once, and mention any price trade-off.",
    },
 
    # ---------------- Cost savings calculations ----------------
    {
        "id": "cost_savings_1",
        "question": "How much would I save each month if I moved my water heater use to off-peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "The response should use peak and off-peak rates from the pricing data, calculate the savings with the calculation tool, show the assumptions used (usage and rates), and give a monthly savings estimate.",
    },
    {
        "id": "cost_savings_2",
        "question": "My home uses about 30 kWh per day and I think I can cut that to 24 kWh. At 15 cents per kWh, how much would that save me per month and per year?",
        "expected_tools": ["calculate_energy_savings"],
        "expected_response": "The response should show 6 kWh saved per day (a 20 percent reduction), about $0.90 per day, about $27 per month (30 days), and about $328.50 per year.",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

## 3. Run Agent Tests

In [9]:
CONTEXT = "Location: San Francisco, CA"

In [10]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )

        tools_used = []
        for msg in response["messages"]:
            obj = msg.model_dump()
            if obj.get("tool_call_id"):
                tools_used.append(msg.name)
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_response': test_case['expected_response'],
            'tools': tools_used,
            'expected_tools': test_case['expected_tools'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_response': test_case['expected_response'],
            'tools': [],
            'expected_tools': test_case['expected_tools'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: ev_charging_2
Question: I need about 30 kWh added to my EV by 7 AM tomorrow. What is the cheapest window tonight, and how much cheaper is it than charging during the evening peak?
--------------------------------------------------

Test 3: thermostat_1
Question: What thermostat settings should I use tomorrow so I stay comfortable but save energy?
--------------------------------------------------

Test 4: thermostat_2
Question: Is it worth pre-cooling my house before the afternoon peak this week?
--------------------------------------------------

Test 5: appliance_scheduling_1
Question: What is the best time tomorrow to run my dishwasher and washing machine?
--------------------------------------------------

Test 6: appliance_scheduling_2
Question: Looking at my recent energy u

In [11]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='Location: San Francisco, CA', additional_kwargs={}, response_metadata={}, id='5ee4358b-6c5c-4c76-ba36-08d83009220d'),
    HumanMessage(content='When should I charge my electric car tomorrow to minimize cost and maximize solar power?', additional_kwargs={}, response_metadata={}, id='0b3bb8a4-1a9d-45ef-8045-b1586e914bfb'),
    AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_M8WLIb28uIr67B5lk9Z28lMi', 'function': {'arguments': '{"location": "San Francisco, CA", "days": 1}', 'name': 'get_weather_forecast'}, 'type': 'function'}, {'id': 'call_6eLta9m5czrtH8z6V0XZ738I', 'function': {'arguments': '{"date": "2023-10-04"}', 'name': 'get_electricity_prices'}, 'type': 'function'}, {'id': 'call_QogvvZKJ0bExrb8WbfTzv4FM', 'function': {'arguments': '{"start_date": "2023-10-04", "end_date": "2023-

## 4. Evaluate Responses

In [12]:
# TODO: Implement evaluation functions
# Create functions to evaluate:
# - Final Response
# - Tool usage

In [13]:
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

In [14]:
load_dotenv()

True

In [15]:
# TODO: Create a response evaluator
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected response"""
    JUDGE_MODEL = "gpt-4o-mini"   # CHECK: any chat model available through your key
    PASS_SCORE = 4                # scores run from 1 to 5; 4 or higher passes
 
    class ResponseEvaluation(BaseModel):
        score: int = Field(description="Rating from 1 (fails the expectations) to 5 (fully meets them)")
        covered_elements: List[str] = Field(default_factory=list, description="Expected elements the response covers")
        missing_elements: List[str] = Field(default_factory=list, description="Expected elements the response is missing")
        reasoning: str = Field(description="One to three sentences explaining the score")
 
    if not final_response or not str(final_response).strip():
        return {
            "score": 1, "passed": False, "covered_elements": [],
            "missing_elements": ["No response was produced"],
            "reasoning": "The agent returned an empty response.", "error": None,
        }
 
    prompt = f"""You are grading the answer of a home energy advisor assistant.
 
                Question asked by the homeowner:
                {question}
                
                What a good response should contain:
                {expected_response}
                
                Assistant's response:
                {final_response}
                
                Grade the response against what a good response should contain.
                Scoring guide:
                5 = covers every expected element, is specific and actionable, and gives numbers or times where expected
                4 = covers the expected elements with minor gaps or some vagueness
                3 = covers some expected elements but misses an important one, or is too generic
                2 = mostly off target, or contains clear errors or unsupported claims
                1 = does not answer the question or fails the expectations
                
                List which expected elements are covered and which are missing.
                Do not reward length. Penalize invented data or numbers presented as facts without a source.
                """
 
    try:
        judge = ChatOpenAI(
            model=JUDGE_MODEL,
            temperature=0,
            base_url="https://openai.vocareum.com/v1",
            api_key=os.getenv("VOCAREUM_API_KEY"),
                      ).with_structured_output(ResponseEvaluation)
        
        verdict = judge.invoke(prompt)
        
        score = max(1, min(5, int(verdict.score)))
        
        return {
            "score": score,
            "passed": score >= PASS_SCORE,
            "covered_elements": list(verdict.covered_elements),
            "missing_elements": list(verdict.missing_elements),
            "reasoning": verdict.reasoning,
            "error": None,
        }

    except Exception as e:
        return {
            "score": None, "passed": False, "covered_elements": [],
            "missing_elements": [], "reasoning": "", "error": str(e),
        }

In [16]:
# TODO: Create a tool udage evaluator
def evaluate_tool_usage(tools_used, expected_tools):
    used = tools_used
    expected = expected_tools

    hits = [t for t in expected if t in used]
    missing = [t for t in expected if t not in used]
    unexpected = [t for t in used if t not in expected]

    recall = len(hits) / len(expected) if expected else 1.0
    if used:
        precision = len(hits) / len(used)
    else:
        precision = 1.0 if not expected else 0.0
 
    return {
        "tools_used": used,
        "expected_tools": expected,
        "missing_tools": missing,
        "unexpected_tools": unexpected,
        "recall": round(recall, 3),
        "precision": round(precision, 3),
        "score": round(recall, 3),      # extra tools are reported but not penalized
        "passed": not missing,
    }

In [17]:
# TODO: Generate a comprehensive evaluation report
# Calculate overall scores and metrics
# Identify strengths and weaknesses
# Provide recommendations for improvement
def generate_evaluation_report(results=None):
    """
    Print an evaluation report and return it as a dict.
 
    `results` is a list with one dict per test case:
        {"id": ..., "response_eval": evaluate_response(...), "tool_eval": evaluate_tool_usage(...)}
    If omitted, a notebook variable named `evaluation_results` is used.   # CHECK
    """

    RESPONSE_WEIGHT, TOOL_WEIGHT = 0.6, 0.4
    TOOL_HINTS = {
        "get_weather_forecast": "tell the agent to check the forecast before giving timing, setpoint or solar advice",
        "get_electricity_prices": "tell the agent to fetch hourly prices before recommending when to run any load",
        "query_energy_usage": "tell the agent to look up the household's actual usage by device before estimating savings",
        "query_solar_generation": "tell the agent to check solar generation data when advising on solar use",
        "get_recent_energy_summary": "tell the agent to start from the recent usage summary for questions about current usage",
        "search_energy_tips": "tell the agent to search the tips knowledge base and base its advice on what it finds",
        "calculate_energy_savings": "tell the agent to use the savings calculator for every savings figure",
    }
 
    if results is None:
        results = globals().get("evaluation_results")
    if not results:
        print("No results to report. Pass a list of results, or store them in `evaluation_results`.")
        return {}
 
    # ---- Per-case scores ----
    rows = []
    for r in results:
        resp, tool = r["response_eval"], r["tool_eval"]
        resp_norm = (resp["score"] - 1) / 4 if resp["score"] is not None else 0.0
        rows.append({
            "id": r.get("id", "case"),
            "score": resp["score"],
            "tool_score": tool["score"],
            "combined": RESPONSE_WEIGHT * resp_norm + TOOL_WEIGHT * tool["score"],
            "passed": resp["passed"] and tool["passed"],
            "judge_error": resp.get("error"),
            "missing_tools": tool["missing_tools"],
            "missing_elements": resp["missing_elements"],
            "reasoning": resp["reasoning"],
            "response_passed": resp["passed"],
            "tool_passed": tool["passed"],
        })
 
    n = len(rows)
    judged = [x for x in rows if x["score"] is not None]
    avg_response = sum(x["score"] for x in judged) / len(judged) if judged else 0.0
    avg_tool = sum(x["tool_score"] for x in rows) / n
    avg_combined = sum(x["combined"] for x in rows) / n
    response_pass_rate = sum(x["response_passed"] for x in rows) / n
    tool_pass_rate = sum(x["tool_passed"] for x in rows) / n
    overall_pass_rate = sum(x["passed"] for x in rows) / n
 
    # ---- Per-category breakdown (category = id without the trailing number) ----
    groups = defaultdict(list)
    for x in rows:
        head, _, tail = x["id"].rpartition("_")
        groups[head if head and tail.isdigit() else x["id"]].append(x)
    by_category = {}
    for cat, xs in groups.items():
        valid = [x for x in xs if not x["judge_error"]]   # judge failures say nothing about the agent
        if valid:
            by_category[cat] = {
                "cases": len(valid),
                "avg_response_score": round(sum(x["score"] for x in valid) / len(valid), 2),
                "avg_tool_score": round(sum(x["tool_score"] for x in valid) / len(valid), 2),
                "avg_combined": round(sum(x["combined"] for x in valid) / len(valid), 2),
            }
 
    missed = Counter(t for x in rows for t in x["missing_tools"])
    judge_errors = [x["id"] for x in rows if x["judge_error"]]
 
    # ---- Strengths and weaknesses ----
    strengths, weaknesses = [], []
    for cat, s in by_category.items():
        if s["avg_combined"] >= 0.8:
            strengths.append(f"Strong on '{cat}' (combined score {s['avg_combined']:.2f})")
        elif s["avg_combined"] < 0.7:
            weaknesses.append(f"Weak on '{cat}' (combined score {s['avg_combined']:.2f})")
    if tool_pass_rate >= 0.9:
        strengths.append(f"Calls the expected tools reliably ({tool_pass_rate:.0%} of cases)")
    elif tool_pass_rate < 0.7:
        weaknesses.append(f"Often skips expected tools (only {tool_pass_rate:.0%} of cases complete)")
    if judged and avg_response >= 4.0:
        strengths.append(f"High response quality (average judge score {avg_response:.1f}/5)")
    elif judged and avg_response < 3.5:
        weaknesses.append(f"Response quality needs work (average judge score {avg_response:.1f}/5)")
    for x in judged:
        if not x["response_passed"]:
            gaps = "; ".join(x["missing_elements"][:3]) or x["reasoning"]
            weaknesses.append(f"{x['id']}: {gaps}")
 
    # ---- Recommendations ----
    recommendations = []
    for tool, count in missed.most_common():
        hint = TOOL_HINTS.get(tool, "instruct the agent when to use this tool")
        recommendations.append(
            f"Update the system prompt to {hint} ('{tool}' was missed in {count} of {n} cases)."
        )
    if judged and avg_response < 4.0:
        recommendations.append(
            "Add stricter recommendation rules to the prompt: name the device, the time window "
            "and the estimated savings in every answer."
        )
    weak_cats = [c for c, s in by_category.items() if s["avg_combined"] < 0.7]
    if weak_cats:
        recommendations.append(
            "Review the transcripts for the weakest categories and add targeted instructions: "
            + ", ".join(weak_cats) + "."
        )
    if judge_errors:
        recommendations.append(
            "The judge model failed on: " + ", ".join(judge_errors)
            + ". Check JUDGE_MODEL and your API key, then re-run."
        )
    if not recommendations:
        recommendations.append(
            "All checks passed. Add harder test cases (missing data, tool failures, off-topic "
            "questions) to keep stress-testing the agent."
        )
 
    # ---- Print the report ----
    line = "=" * 78
    avg_response_text = f"{avg_response:.2f} / 5" if judged else "n/a (judge failed)"
    print(line)
    print("ECOHOME AGENT EVALUATION REPORT")
    print(line)
    print(f"Test cases:                 {n}")
    print(f"Average response score:     {avg_response_text}   (pass rate {response_pass_rate:.0%})")
    print(f"Average tool usage score:   {avg_tool:.2f} / 1   (pass rate {tool_pass_rate:.0%})")
    print(f"Combined score:             {avg_combined:.2f} / 1   "
          f"(weights: response {RESPONSE_WEIGHT:.0%}, tools {TOOL_WEIGHT:.0%})")
    print(f"Cases passing both checks:  {overall_pass_rate:.0%}")
 
    print("\nRESULTS BY TEST CASE")
    print(f"{'id':<26}{'response':>9}{'tools':>8}{'combined':>10}  status")
    for x in rows:
        sc = str(x["score"]) if x["score"] is not None else "err"
        print(f"{x['id']:<26}{sc:>9}{x['tool_score']:>8.2f}{x['combined']:>10.2f}  "
              f"{'PASS' if x['passed'] else 'FAIL'}")
 
    print("\nRESULTS BY CATEGORY")
    print(f"{'category':<26}{'cases':>6}{'response':>10}{'tools':>8}{'combined':>10}")
    for cat, s in by_category.items():
        print(f"{cat:<26}{s['cases']:>6}{s['avg_response_score']:>10.2f}"
              f"{s['avg_tool_score']:>8.2f}{s['avg_combined']:>10.2f}")
 
    print("\nSTRENGTHS")
    for item in strengths or ["None identified yet."]:
        print(f"  + {item}")
    print("\nWEAKNESSES")
    for item in weaknesses or ["None identified."]:
        print(f"  - {item}")
    print("\nRECOMMENDATIONS")
    for i, item in enumerate(recommendations, 1):
        print(f"  {i}. {item}")
    print(line)
 
    return {
        "total_cases": n,
        "avg_response_score_1_to_5": round(avg_response, 2),
        "avg_tool_score": round(avg_tool, 2),
        "avg_combined_score": round(avg_combined, 2),
        "response_pass_rate": round(response_pass_rate, 2),
        "tool_pass_rate": round(tool_pass_rate, 2),
        "overall_pass_rate": round(overall_pass_rate, 2),
        "by_category": by_category,
        "most_missed_tools": dict(missed),
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
    }

In [18]:
response= []
for i in range(10):
    res= test_results[i]
    res_eval= evaluate_response(res['question'], res['response'], res['expected_response'])
    tool_eval= evaluate_tool_usage(res['tools'], res['expected_tools'])

    response.append(
        {
            'id': res['test_id'],
            'response_eval': res_eval,
            'tool_eval': tool_eval
        }
    )

In [19]:
generate_evaluation_report(response)

ECOHOME AGENT EVALUATION REPORT
Test cases:                 10
Average response score:     3.40 / 5   (pass rate 40%)
Average tool usage score:   0.67 / 1   (pass rate 40%)
Combined score:             0.63 / 1   (weights: response 60%, tools 40%)
Cases passing both checks:  30%

RESULTS BY TEST CASE
id                         response   tools  combined  status
ev_charging_1                     4    1.00      0.85  PASS
ev_charging_2                     3    0.50      0.50  FAIL
thermostat_1                      4    0.50      0.65  FAIL
thermostat_2                      3    0.67      0.57  FAIL
appliance_scheduling_1            3    0.50      0.50  FAIL
appliance_scheduling_2            5    1.00      1.00  PASS
solar_maximization_1              3    0.00      0.30  FAIL
solar_maximization_2              4    1.00      0.85  PASS
cost_savings_1                    2    0.50      0.35  FAIL
cost_savings_2                    3    1.00      0.70  FAIL

RESULTS BY CATEGORY
category        

{'total_cases': 10,
 'avg_response_score_1_to_5': 3.4,
 'avg_tool_score': 0.67,
 'avg_combined_score': 0.63,
 'response_pass_rate': 0.4,
 'tool_pass_rate': 0.4,
 'overall_pass_rate': 0.3,
 'by_category': {'ev_charging': {'cases': 2,
   'avg_response_score': 3.5,
   'avg_tool_score': 0.75,
   'avg_combined': 0.68},
  'thermostat': {'cases': 2,
   'avg_response_score': 3.5,
   'avg_tool_score': 0.58,
   'avg_combined': 0.61},
  'appliance_scheduling': {'cases': 2,
   'avg_response_score': 4.0,
   'avg_tool_score': 0.75,
   'avg_combined': 0.75},
  'solar_maximization': {'cases': 2,
   'avg_response_score': 3.5,
   'avg_tool_score': 0.5,
   'avg_combined': 0.57},
  'cost_savings': {'cases': 2,
   'avg_response_score': 2.5,
   'avg_tool_score': 0.75,
   'avg_combined': 0.52}},
 'most_missed_tools': {'calculate_energy_savings': 2,
  'search_energy_tips': 3,
  'get_recent_energy_summary': 1},
 'strengths': [],
 'weaknesses': ["Weak on 'ev_charging' (combined score 0.68)",
  "Weak on 'thermos